In [ ]:
import numpy as np
import  pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tabpfn import TabPFNRegressor
import math

def  mape(y_true, y_pred):
    return np.mean(np.abs((y_pred - y_true) / y_true)) * 100


import warnings
warnings.filterwarnings("ignore")


plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['font.family'] = 'Times New Roman' 
plt.rcParams['axes.unicode_minus'] = False 
plt.rcParams['font.serif'] = ['Times New Roman'] 


In [ ]:
import numpy as np

def create_missing_data(X, missing_rate, random_state=42):

    np.random.seed(random_state)
    
    X_missing = X.copy()
    n_samples, n_features = X.shape
    n_missing = int(n_samples * n_features * missing_rate)
    
    rows = np.random.randint(0, n_samples, n_missing)
    cols = np.random.randint(0, n_features, n_missing)
    for r, c in zip(rows, cols):
        X_missing.iloc[r, c] = np.nan
        
    return X_missing

### TabPFN

In [ ]:

df_data = pd.read_csv('data.csv',sep=',',header=0,index_col=0)

missing_rates = 0.1  # missing_rates

X = df_data.iloc[:,:-5]
X = create_missing_data(X, missing_rates)
y = df_data.iloc[:,-1]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=1/5,    
    random_state=42
)


reg = TabPFNRegressor(ignore_pretraining_limits=True)
reg.fit(X_train, y_train)

TabPFN_y_train_pred = reg.predict(X_train)
print('\nTrain mse:',mean_squared_error(TabPFN_y_train_pred,y_train))
print('Train rmse:',np.sqrt(mean_squared_error(TabPFN_y_train_pred,y_train)))
print('Train mae:',mean_absolute_error(TabPFN_y_train_pred,y_train))
print('Train mape:',mape(TabPFN_y_train_pred,y_train))
print('Train R2:',r2_score(y_train,TabPFN_y_train_pred))

TabPFN_y_test_pred = reg.predict(X_test)
print('\nTest mse:',mean_squared_error(TabPFN_y_test_pred,y_test))
print('Test rmse:',np.sqrt(mean_squared_error(TabPFN_y_test_pred,y_test)))
print('Test mae:',mean_absolute_error(TabPFN_y_test_pred,y_test))
print('Test mape:',mape(TabPFN_y_test_pred,y_test))
print('Test R2:',r2_score(y_test,TabPFN_y_test_pred))



### XGBoost

In [ ]:
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import TimeSeriesSplit


def objective(trial):
    param = {
        'booster': 'gbtree',
        'n_estimators': trial.suggest_int('n_estimators', 100,150),
        'learning_rate': trial.suggest_float('learning_rate', 0.1, 0.25),
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 30),
        'max_depth': trial.suggest_int('max_depth',10, 20),
        'gamma': trial.suggest_float('gamma', 1e-5, 1e-2, log=True),
        'subsample': trial.suggest_float('subsample', 0.8,1),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.25, 0.5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.25, 0.5, log=True),
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'random_state': 0
    }
    
    model = XGBRegressor(**param)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
    return -score

study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=0))

study.optimize(objective, n_trials=100)

best_params = study.best_params
print('best_params:', best_params)



In [ ]:
xgb_reg = XGBRegressor(**best_params, random_state=0)
xgb_reg.fit(X_train,y_train)

XGBoost_y_train_pred = xgb_reg.predict(X_train)
print('\nTrain mse:',mean_squared_error(XGBoost_y_train_pred,y_train))
print('Train rmse:',np.sqrt(mean_squared_error(XGBoost_y_train_pred,y_train)))
print('Train mae:',mean_absolute_error(XGBoost_y_train_pred,y_train))
print('Train mape:',mape(XGBoost_y_train_pred,y_train))
print('Train R2:',xgb_reg.score(X_train,y_train))


XGBoost_y_test_pred = xgb_reg.predict(X_test)
print('\nTest mse:',mean_squared_error(XGBoost_y_test_pred,y_test))
print('Test rmse:',np.sqrt(mean_squared_error(XGBoost_y_test_pred,y_test)))
print('Test mae:',mean_absolute_error(XGBoost_y_test_pred,y_test))
print('Test mape:',mape(XGBoost_y_test_pred,y_test))
print('Test R2:',xgb_reg.score(X_test,y_test))

